# 2.2 ToolRegistry：目录、查重与装配闭环

**学完你能**：实现并测试工具注册表；把它接进 ReAct 循环；（可选）用真实 function calling API 走一次工业协议。

Sources: agent-expert-outline.md 思维 #4 · lecture-notes.md Slide 4–5 · OpenAI function calling 指南

## 概念导读：注册表是模型与工具之间唯一的目录

三条纪律（tests/test_tools.py 已锁定）：
1. **重名即炸**（fail-closed）：同名不同义是事故温床；
2. **未知即报**：`get("nope")` 抛 KeyError，由调用方翻译成模型的 Observation；
3. **一键装配**：`to_callables()` 直接喂 `run_react`——注册表与循环解耦但即插即用。

In [ ]:
# 复现三条纪律（对照 agent/tools/registry.py 与 tests/test_tools.py）
from agent.builtin.calc import calc
from agent.tools.base import tool_from_fn
from agent.tools.registry import ToolRegistry

reg = ToolRegistry()
reg.register(tool_from_fn(calc, "calc", "安全求值算术表达式，如 3*(7+2)。只支持四则与括号。"))
print("目录:", reg.names())
print("提示行渲染:\n" + reg.prompt_lines())
try:
    reg.register(tool_from_fn(calc, "calc", "重复"))
except ValueError as e:
    print("查重生效:", e)

In [ ]:
# 装配闭环：注册表 → ReAct（注入假 LLM 验证管道，零成本）
from agent.react import run_react

script = iter(["Thought: 需要计算\nAction: calc 2**10", "Thought: 1024\nFinish: 1024"])
result = run_react("计算 2**10", reg.to_callables(), llm_fn=lambda m: next(script))
print("答案:", result["answer"], "| 第一步观察:", result["steps"][0]["observation"])

In [ ]:
# 【可选·真调用】工业协议预览：OpenAI 兼容 tools 参数（ch02 W4 的 2.3 之前先尝一口）
# 成本提示：一次真实调用（约 2 次模型往返）。跑之前确认 .env 就位。
import json

from llm_utils import ARK_MODEL, _get_client

resp = _get_client().chat.completions.create(
    model=ARK_MODEL,
    messages=[{"role": "user", "content": "用 calc 工具计算 3*(7+2)"}],
    tools=[{"type": "function", "function": {
        "name": "calc",
        "description": "安全求值算术表达式",
        "parameters": reg.get("calc").json_schema(),
    }}],
)
msg = resp.choices[0].message
print("模型想调用的工具:", json.dumps(msg.model_dump().get("tool_calls", "（直接回答了）"), ensure_ascii=False)[:300])

## 练习（答案只进 solutions 子仓）

1. **改参数**：再注册一个工具（timed_echo 或 fake_weather，exercises 良好档），观察 prompt_lines 的变化——模型看到的目录变长会发生什么？
2. **破坏性实验**：把 to_callables() 的输出喂给 run_react 但删掉注册表里的工具名（模拟目录与实现不同步），验证 [未知工具] 边界依然失败关闭。
3. **扩展题（挑战档预告）**：给 ToolRegistry 加 unregister() 与 with_defaults() 上下文管理器，先写失败测试。

## 提交前检查

```bash
uv run pytest -q && uvx ruff check . && uv run python smoke.py
jupyter nbconvert --clear-output --inplace ch02-tools/2.2-registry-dispatch.ipynb
```